In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import tiktoken
from torch.utils.data import Dataset, DataLoader

### This Notebook Addresses the problem of KV Cache.

Checkout : https://magazine.sebastianraschka.com/p/coding-the-kv-cache-in-llms

In [ ]:
class MultiHeadAttentionCached(nn.Module):
    '''Class Implementation of KV Cache in MHA.'''

    def __init__(self, d_in, d_out, context_length, num_heads, dropout=0.1, qkv_bias=False, kv_cache=False):
        super().__init__()

        assert d_out % num_heads == 0, 'd_out must be divisible by num_heads'
        self.head_dim = d_out // num_heads
        self.num_heads = num_heads
        self.d_in, self.d_out, self.context_length = d_in, d_out, context_length
        self.kv_cache = kv_cache

        self.W_q = nn.Linear(in_features=d_in, out_features=d_out, bias=qkv_bias)
        self.W_k = nn.Linear(in_features=d_in, out_features=d_out, bias=qkv_bias)
        self.W_v = nn.Linear(in_features=d_in, out_features=d_out, bias=qkv_bias)

        self.register_buffer(
            name='mask',
            tensor=torch.triu(input=torch.ones(size=(context_length, context_length)), diagonal=1),
            persistent=False
        )

        self.dropout = nn.Dropout(dropout)
        self.out_proj = nn.Linear(in_features=d_out, out_features=d_out)

        #Newly added components for KV Cache.
        self.register_buffer(
                    name='k_cache',
                    tensor=None,
                    persistent=False
                )
        self.register_buffer(
                            name='v_cache',
                            tensor=None,
                            persistent=False
                        )
        #Added to keep track of which position token for we are decoding / generating.
        self.ptr_current_pos = 0


    def forward(self, x):

        #x or input is of shape (batch_size, seq_length, d_in)
        #Note: If kv_cache=True then num_tokens is num_new_tokens.
        batch_size, num_tokens, d_in = x.shape

        #If kv_cache=True then Q, K_new, V_new size is (batch_size, new_seq_length, d_out)
        Q, K_new, V_new = self.W_q(x), self.W_k(x), self.W_v(x)  # create (batch_size, seq_length, d_in) ->(batch_size, seq_length, d_out)
        # Unroll last dim: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        Q = Q.view(batch_size, num_tokens, self.num_heads, self.head_dim)
        K_new = K_new.view(batch_size, num_tokens, self.num_heads, self.head_dim)
        V_new = V_new.view(batch_size, num_tokens, self.num_heads, self.head_dim)


        ## Newly added KV Cache Part.
        #If we use KV cache we can only pass new tokens and the older tokens will be stored in KV cache format.
        #As we are passing only new tokens so only new queries will be there.
        if self.kv_cache:
            # pass
            if self.k_cache == None:
                self.k_cache, self.v_cache = K_new, V_new
            else:
                #(batch_size, newly_added_seq_length, d_out) --> (batch_size, kv_cached_seq_length + newly_added_seq_length, d_out)
                self.k_cache, self.v_cache = torch.cat([self.k_cache, K_new], dim=1), torch.cat([self.v_cache, V_new], dim=1) #Concating along the num_tokens_dim
            #Once we have got the old Keys & Values concatenated withnew Ones now time for Attention Calculation.
            K, V = self.k_cache, self.v_cache
        else:
            #If no KV cache then we need to pass all tokens everytime to the model .
            K, V = K_new, V_new


        #Now we need to change calculate Attention for each block separately so first we have to interchange the num_tokens & num_heads place.
        Q, K, V = Q.transpose(1,2), K.transpose(1,2), V.transpose(1,2)  #Shape Output: (batch_size, num_heads,  num_tokens, head_dim)

        #If kv_cache == False : #Output Shape: (batch_size, num_heads,  seq_length, seq_length)
        #If kv_cache == True : #Output Shape: (batch_size, num_heads,  Q_num_tokens, K_num_tokens), so if we generate 1 token and pass 1 token then #Output Shape: (batch_size, num_heads,  1, K_num_tokens)
        #As we may generate 1 token at a time with only one newly added token each time, then we have to attend to each and every previous token than this new one.
        attn_scores = Q @ K.transpose(2,3)  


        ##Newly modified part in Causal Masking for KV cache .
        #Older Masking across whole Tokens.
        # attn_scores.masked_fill_(
        #     self.mask.bool()[:num_tokens, :num_tokens], -torch.inf
        # )
        #Here we are using the track keeping flag to do masking from 2th position to 5th position if we are decoding from 2 th position to 5th position.
        Q_num_tokens, K_num_tokens = Q.shape[-2], K.shape[-2] #Q/K shape:(batch_size, num_heads,  Q/K_num_tokens, head_dim)

        if self.kv_cache :
            mask_bool = self.mask.bool()[self.ptr_current_pos:self.ptr_current_pos + Q_num_tokens, : K_num_tokens]
            #Keeping track of last updated postions of the tokens.
            self.ptr_current_pos += Q_num_tokens
        else:
            mask_bool = self.mask.bool()[:Q_num_tokens, :K_num_tokens]


        attn_scores.masked_fill_(mask_bool, -torch.inf)


        attn_weights = torch.softmax(attn_scores/(K.shape[-1]**0.5), dim=-1)  #(batch_size, num_heads,  num_tokens , num_tokens)
        attn_weights = self.dropout(attn_weights)

        context_vectors = attn_weights @ V  #Output Shape: (batch_size, num_heads,  num_tokens, head_dim)
        #Reversing the position of num_heads & num_tokens
        context_vectors =  context_vectors.transpose(1,2)  #Output Shape: (batch_size, num_tokens, num_heads, head_dim)

        #Now we need to make the current context_vector shape memory contiguus and then couple the num_heads & head_dim into one.

        context_vectors = context_vectors.contiguous().view(batch_size, num_tokens, self.d_out)  #Merged the num_heads & head_dim into d_out.

        context_vectors = self.out_proj(context_vectors)
        return context_vectors

    def reset_cache(self):
        '''Function to reset the KV Cache Storage.'''
        self.k_cache, self.v_cache = None, None
        self.ptr_current_pos = 0